# Bước 1: Setup Google Drive & Thư mục

In [ ]:
!pip install torchaudio speechbrain tqdm soundfile\nfrom google.colab import drive
drive.mount('/content/drive')
import os
data_dir = '/content/drive/MyDrive/TSE_Dataset'
os.makedirs(data_dir, exist_ok=True)
os.chdir(data_dir)
print(f'Working directory: {os.getcwd()}')

# Bước 2: Tải & Giải nén các Dataset (Chỉ chạy 1 lần)

In [ ]:
import urllib.request
import tarfile
import zipfile
import os
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

datasets = {
    'train-clean-100': 'https://openslr.trmal.net/resources/12/train-clean-100.tar.gz',
    'train-clean-360': 'https://openslr.trmal.net/resources/12/train-clean-360.tar.gz',
    'musan': 'https://openslr.trmal.net/resources/17/musan.tar.gz',
    'wham_noise': 'https://my-bucket-a8b4b49c25c811ee9a7e8bba05fa24c7.s3.amazonaws.com/wham_noise.zip',
    'rirs_noises': 'https://openslr.trmal.net/resources/28/rirs_noises.zip'
}
for name, url in datasets.items():
    file_name = url.split('/')[-1]
    if not os.path.exists(file_name):
        print(f'\nĐang tải {name}...')
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=name) as t:
            urllib.request.urlretrieve(url, filename=file_name, reporthook=t.update_to)
    else:
        print(f'\n{name} đã được tải.')
    
    print(f'Đang giải nén {name}... (vui lòng đợi)')
    if file_name.endswith('.tar.gz'):
        with tarfile.open(file_name, 'r:gz') as tar:
            tar.extractall()
    elif file_name.endswith('.zip'):
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall()
print('\nHoàn tất tải và giải nén dữ liệu gốc!')

# Bước 3: Mix Audio động và Lưu vào Drive
Đoạn code này áp dụng công thức 70% TSE, 15% Noise, 15% Interferer, và 10% Negative Case (Absent Speaker).

In [ ]:

import os
import random
import torchaudio
import torch
import math
import glob
import json
import shutil
import soundfile as sf
from functools import lru_cache
from tqdm import tqdm

# --- CONFIGURATION ---
SAMPLE_RATE = 16000
MAX_RIR_FILES = 2000
MAX_RIR_SECONDS = 0.5
RESET_OUTPUT_DIR = False
REBUILD_FILE_INDEX = False
CHECKPOINT_EVERY = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
DATA_DIR = "/content/drive/MyDrive/TSE_Dataset"
OUTPUT_DIR = os.path.join(DATA_DIR, "mixed_dataset")
if RESET_OUTPUT_DIR:
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Writing mixed wav files directly to Google Drive: {OUTPUT_DIR}")
print(f"Mixing device: {DEVICE}")

# 1. Quét file (Lấy danh sách wav/flac). Cache để lần sau không phải scan lại Google Drive.
INDEX_PATH = os.path.join(DATA_DIR, "file_index.json")
if os.path.exists(INDEX_PATH) and not REBUILD_FILE_INDEX:
    print(f"Loading cached file index: {INDEX_PATH}")
    with open(INDEX_PATH, "r") as f:
        file_index = json.load(f)
    libri_files = file_index["libri_files"]
    musan_noise = file_index["musan_noise"]
    wham_noise = file_index["wham_noise"]
    rir_files = file_index["rir_files"]
else:
    print("Scanning files and writing cache...")
    libri_files = glob.glob(os.path.join(DATA_DIR, "LibriSpeech", "*", "*", "*", "*.flac"))
    musan_noise = glob.glob(os.path.join(DATA_DIR, "musan", "noise", "*", "*.wav")) +               glob.glob(os.path.join(DATA_DIR, "musan", "music", "*", "*.wav")) +               glob.glob(os.path.join(DATA_DIR, "musan", "speech", "*", "*.wav"))
    wham_noise = glob.glob(os.path.join(DATA_DIR, "cv", "noise", "*.wav")) # Tùy cấu trúc wham
    rir_files = glob.glob(os.path.join(DATA_DIR, "RIRS_NOISES", "simulated_rirs", "*", "*", "*.wav")) +             glob.glob(os.path.join(DATA_DIR, "RIRS_NOISES", "real_rirs_isotropic_noises", "*.wav"))
    file_index = {
        "libri_files": libri_files,
        "musan_noise": musan_noise,
        "wham_noise": wham_noise,
        "rir_files": rir_files,
    }
    with open(INDEX_PATH, "w") as f:
        json.dump(file_index, f)

if len(rir_files) > MAX_RIR_FILES:
    rir_files = random.sample(rir_files, MAX_RIR_FILES)

all_noise = musan_noise + wham_noise

# Nhóm LibriSpeech theo speaker
speakers = {}
for f in libri_files:
    parts = f.replace("\\", "/").split('/')
    spk_id = parts[-3]
    if spk_id not in speakers:
        speakers[spk_id] = []
    speakers[spk_id].append(f)

spk_list = list(speakers.keys())
print(f"Found {len(spk_list)} speakers, {len(all_noise)} noises, {len(rir_files)} RIRs")
if len(spk_list) == 0:
    raise RuntimeError(f"No LibriSpeech files found under {DATA_DIR}/LibriSpeech")
if len(all_noise) == 0:
    raise RuntimeError(f"No MUSAN/WHAM noise files found under {DATA_DIR}")
if len(rir_files) == 0:
    raise RuntimeError(f"No RIR files found under {DATA_DIR}/RIRS_NOISES")

resamplers = {}

def resample_if_needed(wav, sr):
    if sr == SAMPLE_RATE:
        return wav
    if sr not in resamplers:
        resamplers[sr] = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
    return resamplers[sr](wav)

def get_audio_info(path):
    try:
        info = sf.info(path)
        return info.samplerate, info.frames
    except Exception:
        if hasattr(torchaudio, "info"):
            info = torchaudio.info(path)
            return info.sample_rate, info.num_frames
        wav, sr = load_audio_file(path)
        return sr, wav.shape[1]

def load_audio_file(path, frame_offset=0, num_frames=-1):
    try:
        data, sr = sf.read(path, start=frame_offset, frames=num_frames, dtype="float32", always_2d=True)
        if data.shape[0] == 0:
            raise RuntimeError("empty audio segment")
        return torch.from_numpy(data.T.copy()), sr
    except Exception as sf_error:
        try:
            return torchaudio.load(path, frame_offset=frame_offset, num_frames=num_frames)
        except TypeError:
            return torchaudio.load(path)
        except Exception as ta_error:
            raise RuntimeError(f"Could not load audio {path}: soundfile={sf_error}; torchaudio={ta_error}") from ta_error

def get_audio(path, max_len=64000):
    input_sr, total_frames = get_audio_info(path)
    frames_needed = math.ceil(max_len * input_sr / SAMPLE_RATE)
    frame_offset = 0
    num_frames = -1
    if total_frames > frames_needed:
        frame_offset = random.randint(0, total_frames - frames_needed)
        num_frames = frames_needed
    wav, sr = load_audio_file(path, frame_offset=frame_offset, num_frames=num_frames)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = resample_if_needed(wav, sr)
    if wav.shape[1] > max_len:
        start = random.randint(0, wav.shape[1] - max_len)
        wav = wav[:, start:start+max_len]
    elif wav.shape[1] < max_len:
        repeats = math.ceil(max_len / wav.shape[1])
        wav = wav.repeat(1, repeats)[:, :max_len]
    return wav.to(DEVICE)

@lru_cache(maxsize=1024)
def load_rir(rir_path):
    rir_wav, sr = load_audio_file(rir_path)
    rir_wav = resample_if_needed(rir_wav, sr)
    max_rir_len = int(MAX_RIR_SECONDS * SAMPLE_RATE)
    rir_wav = rir_wav[:, :max_rir_len]
    return rir_wav

bad_rir_files = set()

def convolve_rir(audio, rir_path):
    last_error = None
    for _ in range(10):
        if rir_path in bad_rir_files:
            rir_path = random.choice(rir_files)
            continue
        try:
            rir_wav = load_rir(rir_path)
            ch = random.randint(0, rir_wav.shape[0] - 1)
            rir_wav = rir_wav[ch:ch+1].to(audio.device)
            audio = torchaudio.functional.fftconvolve(audio, rir_wav)
            return audio[:, :64000]
        except Exception as e:
            bad_rir_files.add(rir_path)
            last_error = e
            rir_path = random.choice(rir_files)
    raise RuntimeError(f"Failed to load a usable RIR after 10 tries. Last error: {last_error}")

def calc_energy(wav):
    return float((torch.sum(wav ** 2) / wav.shape[1]).clamp_min(1e-12).cpu())

# --- MIXING PIPELINE ---
NUM_SAMPLES = 5000 # Chạy thử 5000 samples
METADATA_PATH = os.path.join(OUTPUT_DIR, "metadata.json")
if not RESET_OUTPUT_DIR and os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata_list = json.load(f)
    start_idx = len(metadata_list)
    print(f"Resuming from sample {start_idx}/{NUM_SAMPLES}")
else:
    metadata_list = []
    start_idx = 0

print("Mixing dataset...")
for i in tqdm(range(start_idx, NUM_SAMPLES), initial=start_idx, total=NUM_SAMPLES):
    try:
        # Chọn Target
        target_spk = random.choice(spk_list)
        target_files = speakers[target_spk]
        if len(target_files) < 2: continue
        target_path = random.choice(target_files)
        enroll_path = random.choice([f for f in target_files if f != target_path])
        
        target_wav = get_audio(target_path)
        enroll_wav = get_audio(enroll_path)
        
        # RIR
        rir_path = random.choice(rir_files)
        target_reverb = convolve_rir(target_wav, rir_path)
        target_energy = calc_energy(target_reverb)
        
        # PHÂN LOẠI CASE (10% cơ hội cho Negative Case)
        rand_case = random.random()
        is_negative = rand_case < 0.10
        
        if is_negative:
            # --- NEGATIVE CASE ---
            # Mixture không có Target
            mixture = torch.zeros_like(target_reverb)
            # Target = 0 để model học cách dập âm
            target_reverb = torch.zeros_like(target_reverb)
            case_name = "absent_speaker"
        else:
            # --- POSITIVE CASE ---
            mixture = target_reverb.clone()
            case_name = "full_tse" if rand_case < 0.7 else ("target_noise" if rand_case > 0.85 else "target_interferer")
        
        # Thêm Interferer
        # Negative Case -> 100% có Interferer. Positive Case -> 85% có Interferer.
        if is_negative or rand_case < 0.85:
            interf_spk = random.choice([s for s in spk_list if s != target_spk])
            interf_path = random.choice(speakers[interf_spk])
            interf_wav = get_audio(interf_path)
            interf_reverb = convolve_rir(interf_wav, random.choice(rir_files))
            
            sir_db = random.uniform(-5, 10)
            interf_energy = calc_energy(interf_reverb)
            scale_sir = math.sqrt(target_energy / (interf_energy * (10 ** (sir_db / 10)) + 1e-8))
            mixture += interf_reverb * scale_sir
            
        # Thêm Noise
        # Negative Case -> 100% có Noise. Positive Case -> 85% có Noise.
        if is_negative or rand_case < 0.70 or rand_case > 0.85:
            noise_path = random.choice(all_noise)
            noise_wav = get_audio(noise_path)
            
            snr_db = random.uniform(-5, 15)
            noise_energy = calc_energy(noise_wav)
            scale_snr = math.sqrt(target_energy / (noise_energy * (10 ** (snr_db / 10)) + 1e-8))
            mixture += noise_wav * scale_snr
            
        # Chống Clip
        max_amp = torch.max(torch.abs(mixture))
        if max_amp > 0.9:
            mixture = mixture * (0.9 / max_amp)
            if not is_negative:
                target_reverb = target_reverb * (0.9 / max_amp)
            enroll_wav = enroll_wav * (0.9 / max_amp)
            
        # Lưu file
        mix_name = f"mix_{i:05d}.wav"
        tgt_name = f"tgt_{i:05d}.wav"
        enl_name = f"enl_{i:05d}.wav"
        
        torchaudio.save(os.path.join(OUTPUT_DIR, mix_name), mixture.cpu(), SAMPLE_RATE)
        torchaudio.save(os.path.join(OUTPUT_DIR, tgt_name), target_reverb.cpu(), SAMPLE_RATE)
        torchaudio.save(os.path.join(OUTPUT_DIR, enl_name), enroll_wav.cpu(), SAMPLE_RATE)
        
        # Caching metadata (Thêm trường is_negative để map với train_personalized.py)
        metadata_list.append({
            "mixture": mix_name,
            "target": tgt_name,
            "enrollment": enl_name,
            "target_speaker": target_spk,
            "target_utterance": target_path.split("/")[-1],
            "enroll_utterance": enroll_path.split("/")[-1],
            "case": case_name,
            "is_negative": is_negative
        })

        if len(metadata_list) % CHECKPOINT_EVERY == 0:
            with open(METADATA_PATH, "w") as f:
                json.dump(metadata_list, f, indent=4)
    except Exception as e:
        print(f"Error at sample {i}: {type(e).__name__}: {e}")
        raise

with open(METADATA_PATH, "w") as f:
    json.dump(metadata_list, f, indent=4)

print(f"Hoàn tất tạo Dataset trên Google Drive: {OUTPUT_DIR}")
